In [72]:
#Euro
import os
import sys
import re
import numpy as np
import pandas as pd
import camelot
from PyPDF2 import PdfReader
from openpyxl import load_workbook
import io
import unicodedata
import pdfplumber, re, statistics, pandas as pd


In [122]:
# pip install pdfplumber pandas
import pdfplumber, re, statistics, pandas as pd

pdf_path = "Remittance_euro.pdf"
results = []

# --- Funciones de limpieza y extracción ---
def prefer_doc(doccol, co, detalle):
    """
    Busca prefijo PMP/DEC/PM/DE + número en cualquier lugar de la fila
    """
    full_txt = ' '.join(filter(None, [doccol, co, detalle])).replace('\n', ' ')
    prefix_match = re.search(r'\b(PMP|DEC|PM|DE|DEV)\b', full_txt, flags=re.IGNORECASE)
    num_match    = re.search(r'\b(\d{3,10})\b', full_txt.replace(',', ''))
    if prefix_match and num_match:
        return f"{prefix_match.group(1).upper()} {num_match.group(1)}"
    if num_match:
        return num_match.group(1)
    return ''

def clean_co(co_raw):
    if not co_raw: return ''
    toks = str(co_raw).split()
    for t in toks:
        if re.match(r'^[A-Za-z]{2,5}$', t):
            return t
    return toks[0] if toks else ''

def clean_detalle(co_raw, doccol, detalle_raw, doc_found):
    comb = str(detalle_raw or '').replace('\n', ' ').strip()
    if doc_found:
        comb = re.sub(r'\b' + re.escape(doc_found) + r'\b', ' ', comb)
    comb = re.sub(r'\b\d{3,10}\b', ' ', comb)
    comb = re.sub(r'\s+', ' ', comb).strip()
    return comb

def extract_num_columns(row, num_cols=['Descuentos','Retenciones','Valor Factura','Valor Pago']):
    """
    Extrae todos los números de la fila (incluyendo $-) y los asigna en orden a num_cols
    """
    text = ' '.join([str(row.get(col,'')).replace('\n',' ').replace('\r',' ').strip() for col in num_cols])
    # Buscar números con o sin signo, incluyendo $-
    nums = re.findall(r'\$?-?\d[\d,\.]*', text)
    nums = [n.replace('$','').replace(' ','') for n in nums]
    result = {}
    for i, col in enumerate(num_cols):
        result[col] = nums[i] if i < len(nums) else ''
    return result

def merge_doc_prefix_rows(parsed, doc_field='Doc.Cruce'):
    merged = []
    i = 0
    while i < len(parsed):
        row = parsed[i].copy()
        doc = str(row.get(doc_field, '')).strip()
        if re.match(r'^[A-Za-z]{2,4}$', doc, flags=re.IGNORECASE) and i + 1 < len(parsed):
            nxt = parsed[i+1]
            nxt_doc = str(nxt.get(doc_field, '')).strip()
            if re.match(r'^\d{3,10}$', nxt_doc):
                for k in list(row.keys()):
                    if k == 'page':
                        continue
                    row[k] = (str(row.get(k, '')).strip() + ' ' + str(nxt.get(k, '')).strip()).strip()
                try:
                    row['raw_top'] = min(float(row.get('raw_top', 0)), float(nxt.get('raw_top', row.get('raw_top', 0))))
                except Exception:
                    pass
                merged.append(row)
                i += 2
                continue
        merged.append(row)
        i += 1
    return merged

# --- Lectura del PDF ---
with pdfplumber.open(pdf_path) as pdf:
    for pnum, page in enumerate(pdf.pages, start=1):
        words = page.extract_words()
        if not words:
            continue

        # Detectar encabezados
        header_cands = []
        for w in words:
            t = re.sub(r'\s+', ' ', w['text']).strip()
            tl = t.lower().replace(' ', '')
            if tl in ('reg','reg.') or 'detalle' in tl or 'doc' in tl or 'cruce' in tl or tl in ('c','c.','o','o.') \
               or tl in ('descuentos','retenciones','valorfactura','valorpago'):
                header_cands.append({'text': t, 'x0': w['x0'], 'top': w['top']})
        if not header_cands:
            continue

        # Agrupar encabezados por top
        header_cands.sort(key=lambda x: x['top'])
        clusters = []
        tol = 4
        for h in header_cands:
            if not clusters or abs(h['top'] - clusters[-1]['top_mean']) > tol:
                clusters.append({'items':[h], 'top_mean': h['top']})
            else:
                clusters[-1]['items'].append(h)
                clusters[-1]['top_mean'] = statistics.mean([it['top'] for it in clusters[-1]['items']])
        cluster = max(clusters, key=lambda c: c['top_mean'])
        header_items = sorted(cluster['items'], key=lambda it: it['x0'])
        header_top = cluster['top_mean']

        # Calcular posiciones de columnas
        positions = {}
        co_x_list, doc_x_list = [], []
        for it in header_items:
            tnorm = it['text'].strip().lower().replace(' ','')
            if 'reg' in tnorm:
                positions['Reg.'] = it['x0']
            elif 'detalle' in tnorm:
                positions['Detalle'] = it['x0']
            elif tnorm in ('c','c.','o','o.'):
                co_x_list.append(it['x0'])
            elif 'doc' in tnorm or 'cruce' in tnorm:
                doc_x_list.append(it['x0'])
            elif 'descuentos' in tnorm:
                positions['Descuentos'] = it['x0']
            elif 'retenciones' in tnorm:
                positions['Retenciones'] = it['x0']
            elif 'valorfactura' in tnorm:
                positions['Valor Factura'] = it['x0']
            elif 'valorpago' in tnorm:
                positions['Valor Pago'] = it['x0']

        if not (positions.get('Reg.') and positions.get('Detalle') and doc_x_list):
            continue

        positions['C.O.'] = statistics.mean(co_x_list) if co_x_list else (positions['Reg.'] + min(doc_x_list))/2
        positions['Doc.Cruce'] = statistics.mean(doc_x_list)

        centers_sorted = sorted(positions.items(), key=lambda x: x[1])
        colnames = [c[0] for c in centers_sorted]
        xs = [c[1] for c in centers_sorted]

        # Crear bounds de columnas
        bounds = [0.0] + [(a+b)/2.0 for a,b in zip(xs, xs[1:])] + [page.width + 1.0]
        padding = 5
        for i, col in enumerate(colnames):
            if col in ('Descuentos','Retenciones','Valor Factura','Valor Pago'):
                bounds[i] -= padding
                bounds[i+1] += padding

        # Palabras debajo del encabezado
        data_words = [w for w in words if w['top'] > header_top - 2]
        data_words.sort(key=lambda w: (w['top'], w['x0']))

        # Agrupar por fila
        rows = []
        current, last_top = [], None
        row_tol = 8
        for w in data_words:
            if last_top is None or abs(w['top'] - last_top) <= row_tol:
                current.append(w)
                last_top = w['top'] if last_top is None else (last_top + w['top'])/2.0
            else:
                rows.append(current)
                current = [w]
                last_top = w['top']
        if current:
            rows.append(current)

        # Asignar palabras a columnas
        parsed = []
        for row_words in rows:
            cells = {name: [] for name in colnames}
            for w in row_words:
                x = w['x0']
                col_idx = next((i for i in range(len(bounds)-1) if x >= bounds[i] and x < bounds[i+1]), None)
                if col_idx is None:
                    col_idx = min(range(len(bounds)-1), key=lambda i: abs((bounds[i]+bounds[i+1])/2 - x))
                cells[colnames[col_idx]].append(w['text'])
            row_dict = {'page': pnum, 'raw_top': statistics.mean([w['top'] for w in row_words])}
            for name in colnames:
                row_dict[name] = ' '.join(cells[name]).replace('\n',' ').replace('\r',' ').strip()
            parsed.append(row_dict)

        # Unir filas donde el prefijo PMP/DEC está separado
        parsed = merge_doc_prefix_rows(parsed, doc_field='Doc.Cruce')

        # Filtrar filas válidas
        finals = []
        for r in parsed:
            reg_val = str(r.get('Reg.', '')).strip()
            co_val = str(r.get('C.O.', '')).strip()
            doc_val = str(r.get('Doc.Cruce', '')).strip()
            if re.match(r'^\s*\d+\b', reg_val) or re.match(r'^\s*\d+\b', co_val):
                finals.append(r)
                continue
            if re.search(r'\b([A-Za-z]{2,4})\s*\d{3,10}\b', doc_val):
                finals.append(r)
                continue
            if re.search(r'\b(PMP|PM|DEC|DE)\b', doc_val, flags=re.IGNORECASE):
                finals.append(r)
                continue

        # Corregir caso C.O.
        for r in finals:
            if not re.match(r'^\d+\b', str(r.get('Reg.','')).strip()) and re.match(r'^\d+\b', str(r.get('C.O.','')).strip()):
                m = re.match(r'^(\d+)\s*(.*)$', r['C.O.'])
                if m:
                    r['Reg.'] = m.group(1)
                    r['C.O.'] = m.group(2).strip()

        results.extend(finals)

# --- Paso final: crear DataFrame ---
df = pd.DataFrame(results)

# Unir filas multilínea
grouped_rows = []
buffer = None
num_cols = ['Descuentos','Retenciones','Valor Factura','Valor Pago']
for r in df.to_dict(orient='records'):
    reg_val = r.get('Reg.', '').strip()
    co_val = r.get('C.O.', '').strip()
    if reg_val or co_val:
        if buffer:
            grouped_rows.append(buffer)
        buffer = r.copy()
    else:
        if buffer is None:
            buffer = r.copy()
        else:
            for col in ['Detalle','Doc.Cruce','C.O.'] + num_cols:
                buffer[col] = (buffer.get(col,'') + ' ' + r.get(col,'')).strip()
if buffer:
    grouped_rows.append(buffer)

# Crear DataFrame final con números correctamente extraídos
final_rows = []
for r in grouped_rows:
    reg = str(r.get('Reg.', '')).strip()
    co_raw = r.get('C.O.', '') or ''
    doccol = r.get('Doc.Cruce', '') or ''
    detalle_raw = r.get('Detalle', '') or ''
    doc = prefer_doc(doccol, co_raw, detalle_raw)
    co = clean_co(co_raw)
    detalle = clean_detalle(co_raw, doccol, detalle_raw, doc)
    nums = extract_num_columns(r, num_cols)
    

    final_rows.append({
#        'page': r.get('page', ''),
        'Reg.': reg,
        'C.O.': co,
        'Doc.Cruce': doc,
        'Detalle': detalle,
        'Descuentos': nums['Descuentos'],
        'Retenciones': nums['Retenciones'],
        'Valor Factura': nums['Valor Factura'],
        'Valor Pago': nums['Valor Pago']
    })
    
df_final = pd.DataFrame(final_rows)
mask = (~df_final["Descuentos"].str.contains(',', na=False)) & (df_final["Retenciones"].str.contains(',', na=False))
df_final.loc[mask, ["Descuentos", "Retenciones"]] = df_final.loc[mask, ["Retenciones", "Descuentos"]].values

# --- Guardar CSV ---
out = "Remittance_extracted_table.csv"
df_final.to_csv(out, index=False)
print("Guardado en:", out)
print("Filas extraídas:", len(df_final))
display(df_final.head(30))


Guardado en: Remittance_extracted_table.csv
Filas extraídas: 30


,Reg.,C.O.,Doc.Cruce,Detalle,Descuentos,Retenciones,Valor Factura,Valor Pago
0,1,SAL,DEC 9493,INV 31 ENERO,"255,322",0,"255,322","-255,322"
1,2,SAL,DEC 9494,INV 31 ENERO,"255,259",0,"255,259","-255,259"
2,3,CED,PMP 1180287,INV 31 ENERO,"105,587,423",0,"105,856,505","105,856,505"
3,4,CED,PMP 1179878,INV 31 ENERO,"122,003,557",0,"123,028,526","123,028,526"
4,5,CED,PMP 1179880,INV 31 ENERO,"140,658,680",0,"141,563,421","141,563,421"
5,6,CED,PMP 1185375,INV 31 ENERO,"50,670,419",0,"50,670,419","50,670,419"
6,7,CED,PMP 1185378,INV 31 ENERO,"3,174,301",0,"3,174,301","3,174,301"
7,8,CED,PMP 118537,INV 31 ENERO,"2,032,620",0,"2,032,620","2,032,620"
8,9,CED,PMP 1185379,INV 31 ENERO,"28,719,005",0,"28,719,005","28,719,005"
9,10,CED,PMP 1185377,INV 31 ENERO,"527,999",0,"527,999","527,999"
